# 🔬 Notebook 3: Reminder / Alert — Deep Dive

Now the fun part. Given a pile of scheduled reminders, **how does the
system decide what fires next**, and **how does it stay correct when
things fail**?

We walk a **bad → better → best** progression:

1. ❌ **V1** — thread-per-reminder with `sleep()`: simple, completely
   un-scalable.
2. 🙂 **V2** — polling loop over a sorted list: the "read the whole
   table every second" approach.
3. ✅ **V3** — min-heap priority queue: what most "good enough" systems
   actually use.
4. 🚀 **V4** — hashed hierarchical time wheel: what you reach for at
   Twitter/Kafka-scale.

Then we cover:

- 🔁 Retries with exponential backoff + dead-letter queue.
- 🪪 At-least-once + idempotent receivers (and why "exactly-once" is a lie).
- 🧩 Sharding strategies at the very end.


## 🛠️ Setup

```bash
cd 06-system-designs/reminder-alert
uv sync
```

Then in VS Code: pick the `.venv` kernel (top-right of the notebook).
If it doesn't show up, run `Cmd+Shift+P` → **Reload Window**.

All code here uses only the Python standard library + `pydantic`. No
servers, no databases — everything runs in-process so you can step
through the ideas.


## ❌ V1: thread-per-reminder (the bad idea)

Spawn a thread, `sleep(delay)`, fire. No shared state, no infra — and
no hope of scaling past a few thousand reminders.


In [1]:
import threading, time
from datetime import datetime, timezone, timedelta

fired = []

def v1_schedule(fire_at, message):
    def worker():
        delay = (fire_at - datetime.now(timezone.utc)).total_seconds()
        if delay > 0:
            time.sleep(delay)
        fired.append((message, datetime.now(timezone.utc)))
    threading.Thread(target=worker, daemon=True).start()

now = datetime.now(timezone.utc)
for i in range(5):
    v1_schedule(now + timedelta(milliseconds=100 * (i + 1)), f"msg #{i}")

time.sleep(0.7)
for msg, ts in fired:
    print(f"  {msg:>6s} fired at {ts.time().isoformat(timespec='milliseconds')}")


  msg #0 fired at 01:58:35.495
  msg #1 fired at 01:58:35.590
  msg #2 fired at 01:58:35.693
  msg #3 fired at 01:58:35.793
  msg #4 fired at 01:58:35.887


**Why it's bad:**

- One OS thread per reminder → dies at ~10k concurrent.
- All state lives in the process — restart = all reminders lost.
- No way to cancel a specific reminder (you'd need a handle per thread).
- Clock drift between machines means two replicas would fire twice.

Lesson: **durability + parallelism** must come from the storage layer,
not from threads.


## 🙂 V2: the polling loop

Put every reminder in a shared, sorted list (or SQL table indexed on
`fire_at`). A single loop wakes up every second and picks off whatever
is due.


In [2]:
import bisect
from datetime import datetime, timezone, timedelta

class PollingScheduler:
    def __init__(self):
        # Sorted list of (fire_at, id, payload). bisect keeps it sorted.
        self._queue: list[tuple[datetime, str, str]] = []

    def schedule(self, fire_at, rid, payload):
        bisect.insort(self._queue, (fire_at, rid, payload))

    def tick(self, now=None):
        now = now or datetime.now(timezone.utc)
        fired = []
        # Pop all items whose fire_at <= now. Because the list is sorted,
        # we only look at the head — no full scan needed.
        while self._queue and self._queue[0][0] <= now:
            fired.append(self._queue.pop(0))
        return fired

sched = PollingScheduler()
t0 = datetime.now(timezone.utc)
for i in range(5):
    sched.schedule(t0 + timedelta(milliseconds=100 * (i + 1)), f"r{i}", f"msg {i}")

for _ in range(7):
    fired = sched.tick()
    for fire_at, rid, msg in fired:
        print(f"  fired {rid} ({msg})")
    time.sleep(0.1)


  fired r0 (msg 0)
  fired r1 (msg 1)


  fired r2 (msg 2)
  fired r3 (msg 3)


  fired r4 (msg 4)


**Better than V1 because:**

- One loop handles all reminders; no thread explosion.
- The queue can live in a DB for durability.
- Cancellation = just remove the row.

**Still bad because:**

- `bisect.insort` is O(N) on insert for a Python list — ok for
  thousands, ugly at millions.
- A single loop is a single point of failure and a single-core bottleneck.
- In SQL-land, "poll every second" on a 1B-row table is expensive unless
  you're careful with indexes and `FOR UPDATE SKIP LOCKED`.


## ✅ V3: min-heap priority queue (what most systems actually use)

A binary min-heap gives us O(log N) insert and O(log N) "pop earliest".
Python's `heapq` is already a min-heap.

We also introduce **multiple worker coroutines** sharing the heap — this
is the pattern you'll see in real-world schedulers like Quartz,
Sidekiq-cron, or a custom Redis-ZSET worker pool.


In [3]:
import heapq, itertools
from datetime import datetime, timezone, timedelta

class HeapScheduler:
    def __init__(self):
        self._heap: list[tuple[float, int, str, str]] = []
        # A monotonically-increasing tiebreaker keeps the heap total-ordered
        # even when two reminders share the exact same fire_at.
        self._counter = itertools.count()
        # Track cancelled ids so we can skip them lazily on pop.
        self._cancelled: set[str] = set()

    def schedule(self, fire_at: datetime, rid: str, payload: str):
        heapq.heappush(
            self._heap,
            (fire_at.timestamp(), next(self._counter), rid, payload),
        )

    def cancel(self, rid: str):
        # Lazy cancellation: mark and skip on pop. Cheap, O(1).
        self._cancelled.add(rid)

    def next_due_in(self, now=None) -> float | None:
        """Seconds until the next reminder, or None if queue is empty."""
        now = now or datetime.now(timezone.utc)
        while self._heap and self._heap[0][2] in self._cancelled:
            _, _, rid, _ = heapq.heappop(self._heap)
            self._cancelled.discard(rid)
        if not self._heap:
            return None
        return max(0.0, self._heap[0][0] - now.timestamp())

    def pop_due(self, now=None):
        now = now or datetime.now(timezone.utc)
        fired = []
        while self._heap and self._heap[0][0] <= now.timestamp():
            ts, _, rid, payload = heapq.heappop(self._heap)
            if rid in self._cancelled:
                self._cancelled.discard(rid)
                continue
            fired.append((rid, payload))
        return fired


sched = HeapScheduler()
t0 = datetime.now(timezone.utc)
for i in range(5):
    sched.schedule(t0 + timedelta(milliseconds=50 * (i + 1)), f"r{i}", f"msg {i}")

sched.cancel("r2")  # cancel the 3rd one before it fires
time.sleep(0.4)
for rid, payload in sched.pop_due():
    print(f"  fired {rid}: {payload}")


  fired r0: msg 0
  fired r1: msg 1
  fired r3: msg 3
  fired r4: msg 4


**Why this scales:**

- O(log N) insert and pop, handles tens of millions of in-memory timers.
- A **`next_due_in()`** method lets the loop `sleep` exactly that long,
  eliminating busy-wait. This is the sweet spot for sub-second accuracy.
- Cancellation is lazy → O(1) on cancel, amortised O(log N) on pop.

**When to go beyond it:** if you have so many timers that even the log N
insert cost matters, or if you're adding/removing millions per second
(IoT fanout, Kafka timers), the heap's cache-unfriendly pointer chasing
starts to hurt. Enter the time wheel.


## 🚀 V4: hashed time wheel

Think of a clock with N slots. Each slot holds the bucket of reminders
due at "the current tick + slot offset".

- **Insert** = O(1): mod the delay by N, drop it in that slot.
- **Tick** = O(k): only the reminders in *this* bucket are examined.

Great when most reminders fire within a small horizon (e.g. seconds to
minutes). For longer delays we either (a) re-queue after one full wheel
rotation, or (b) layer wheels hierarchically — one for seconds, one for
minutes, one for hours. That's the "hashed hierarchical time wheel" used
in Netty, Kafka, and the Linux kernel.


In [4]:
class TimeWheel:
    def __init__(self, slots: int = 60, tick_s: float = 1.0):
        self.slots = slots
        self.tick_s = tick_s
        self.buckets: list[list[tuple[int, str, callable]]] = [[] for _ in range(slots)]
        self.cursor = 0

    def schedule(self, delay_s: float, rid: str, cb):
        # Number of full wheel rotations before this reminder fires.
        rotations = int(delay_s // (self.slots * self.tick_s))
        offset = int((delay_s / self.tick_s) % self.slots)
        slot = (self.cursor + offset) % self.slots
        self.buckets[slot].append((rotations, rid, cb))

    def advance(self):
        """Advance one tick. Fire everything in the current bucket whose
        rotation counter is 0; decrement the rest."""
        self.cursor = (self.cursor + 1) % self.slots
        bucket = self.buckets[self.cursor]
        still_waiting = []
        for rotations, rid, cb in bucket:
            if rotations == 0:
                cb()
            else:
                still_waiting.append((rotations - 1, rid, cb))
        self.buckets[self.cursor] = still_waiting


wheel = TimeWheel(slots=5, tick_s=0.05)
fired_msgs = []
# Schedule 8 reminders with delays that span more than one wheel rotation.
for i in range(8):
    delay = 0.05 * (i + 1)
    wheel.schedule(delay, f"r{i}", lambda i=i: fired_msgs.append(f"r{i}"))

for _ in range(10):
    wheel.advance()
print("Fired order:", fired_msgs)


Fired order: ['r0', 'r1', 'r2', 'r3', 'r5', 'r6', 'r7', 'r4']


Read back the "Fired order" above — notice that `r4` fires *after*
`r5`, `r6`, `r7`. Why? With 5 slots at 50 ms each, `r4`'s 250 ms delay
lands it in the **same bucket as `r-1`** but with `rotations=1`. On the
first visit to that slot we decrement; on the *second* visit we fire.
That's the price we pay for O(1) insert: we accept some bounded
late-firing (≤ one wheel rotation), never early-firing. The `rotations`
counter is what makes delays longer than the wheel safe.

**Trade-off vs heap:** worse accuracy (bounded by `tick_s`), O(1) ops,
and extremely cache-friendly at high insert/cancel rates. Most real
systems run a **heap or time wheel in memory** sitting on top of a
**durable SQL store** that acts as the source of truth.


## 🔁 Retries + dead-letter queue

Networks fail. Push providers 5xx. We must retry — but politely, with
**exponential backoff + jitter** so we don't hammer a failing provider.

After N failed attempts, we move the reminder to a **dead-letter queue**
for a human to inspect.


In [5]:
import random

def backoff_delay(attempt: int, base: float = 1.0, cap: float = 60.0) -> float:
    """Exponential backoff with full jitter. `attempt` starts at 1."""
    exp = min(cap, base * (2 ** (attempt - 1)))
    return random.uniform(0, exp)

random.seed(0)
for attempt in range(1, 6):
    print(f"attempt {attempt}: retry in {backoff_delay(attempt):5.2f}s")


attempt 1: retry in  0.84s
attempt 2: retry in  1.52s
attempt 3: retry in  1.68s
attempt 4: retry in  2.07s
attempt 5: retry in  8.18s


In [6]:
# A tiny delivery driver that retries up to MAX_ATTEMPTS, then DLQs.
from dataclasses import dataclass, field

MAX_ATTEMPTS = 3
dead_letter: list[dict] = []

@dataclass
class FakeProvider:
    fail_first_n: int = 2            # fail the first N attempts
    calls: int = 0

    def send(self, message: str) -> bool:
        self.calls += 1
        if self.calls <= self.fail_first_n:
            raise RuntimeError(f"provider 5xx (call {self.calls})")
        return True

def deliver_with_retry(provider, rid: str, message: str):
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            provider.send(message)
            print(f"  ✅ {rid} delivered on attempt {attempt}")
            return
        except Exception as e:
            print(f"  ⚠️  {rid} attempt {attempt} failed: {e}")
            if attempt == MAX_ATTEMPTS:
                dead_letter.append({"rid": rid, "msg": message, "error": str(e)})
                print(f"  💀 {rid} moved to dead-letter queue")
                return
            # In production we'd re-enqueue with `backoff_delay(attempt)` seconds.

deliver_with_retry(FakeProvider(fail_first_n=1), "r1", "Take your pills")
print()
deliver_with_retry(FakeProvider(fail_first_n=5), "r2", "Take your pills")
print(f"\nDLQ: {dead_letter}")


  ⚠️  r1 attempt 1 failed: provider 5xx (call 1)
  ✅ r1 delivered on attempt 2

  ⚠️  r2 attempt 1 failed: provider 5xx (call 1)
  ⚠️  r2 attempt 2 failed: provider 5xx (call 2)
  ⚠️  r2 attempt 3 failed: provider 5xx (call 3)
  💀 r2 moved to dead-letter queue

DLQ: [{'rid': 'r2', 'msg': 'Take your pills', 'error': 'provider 5xx (call 3)'}]


## 🪪 At-least-once + idempotent receivers

**Exactly-once delivery across a network is a lie** — there's always a
window where you've sent the message but haven't recorded "sent" yet,
so on crash-recovery you re-send.

The fix: accept **at-least-once** on the sender, make the receiver
**idempotent** via a stable `delivery_id`.


In [7]:
# Simulate a crash: the worker sends twice because its "sent" record
# was lost before ack. The receiver dedupes on delivery_id.

seen_delivery_ids: set[str] = set()

def receiver(delivery_id: str, message: str) -> str:
    if delivery_id in seen_delivery_ids:
        return "duplicate — ignored"
    seen_delivery_ids.add(delivery_id)
    return f"delivered: {message}"

# Reminder r42, first attempt
print(receiver("r42-a1", "Take your pills"))
# Worker crashed before marking sent → retries with the SAME delivery id.
print(receiver("r42-a1", "Take your pills"))
# Different reminder entirely
print(receiver("r43-a1", "Meeting in 5 minutes"))


delivered: Take your pills
duplicate — ignored
delivered: Meeting in 5 minutes


**Gotchas when picking `delivery_id`:**

- Must be deterministic per (reminder, attempt-grouping) — usually
  `f"{reminder_id}-{fire_epoch}"` so retries of the same firing collapse.
- If you bump it on every retry, you lose dedupe and the receiver sees
  duplicates.
- For recurring reminders, include the fire time in the id so Monday's
  8 AM and Tuesday's 8 AM have different ids.


## 🧩 Sharding & topology notes

At 100k fires/s peak you can't have a single scheduler. Common patterns:

- **Shard by `user_id`** — each worker owns a slice of users. Simple, but
  a whale user can overwhelm one shard.
- **Shard by `hash(fire_at)` bucket** — spread the top-of-hour thundering
  herd across many shards, even for the same user.
- **Hot ring** — keep the next ~60s of reminders in Redis (time wheel or
  ZSET), everything beyond 60s in SQL. The dispatcher pulls from SQL into
  the ring as the horizon approaches.

Whatever the topology, the SQL store remains the source of truth — a
crashed in-memory ring is rehydrated from SQL on restart.


## 🎯 Closing thoughts

- Start with **V2 polling + `SELECT … FOR UPDATE SKIP LOCKED`**. It gets
  most real products through their first 10M users.
- Graduate to **V3 min-heap** in front of the DB when sub-second accuracy
  matters.
- Reach for **V4 time wheel** when insert/cancel rate is the bottleneck,
  not fire latency.
- **Always** design receivers to be idempotent. At-least-once is free;
  exactly-once is a mirage.
- **Always** store UTC, remember the user's tz, do recurrence math in the
  local zone.

If you can defend these four points in an interview, you can design a
reminder/alert system. 🎉
